<a href="https://colab.research.google.com/github/kaustubh8salunkhe/FlyRank-Repo/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaustubh8salunkhe/FlyRank-Repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


**Ans**: *I am declaring a Freestyle lane focused on predicting content decay before it happens. Instead of relying on a hardcoded rule like trend_direction, this project will predict actual observed traffic drops in a future time window.*

In [10]:
import pandas as pd
import os

# Define the directory and file path
data_dir = "data/raw"
file_path = os.path.join(data_dir, "content_refresh_anonymized.csv")

# Check if the directory exists, if not, create it
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f"Created directory: {data_dir}")

# Check if the file exists, if not, download it
if not os.path.exists(file_path):
    print(f"Downloading {os.path.basename(file_path)}...")
    # Raw GitHub URL for the dataset
    github_raw_url = "https://raw.githubusercontent.com/kaustubh8salunkhe/FlyRank-Repo/main/data/raw/content_refresh_anonymized.csv"
    !wget -q $github_raw_url -O $file_path
    print(f"Downloaded {os.path.basename(file_path)} to {file_path}")

# Load the starter dataset
df = pd.read_csv(file_path)

# Calculate how many pages have a negative trend (experiencing decay)
# Note: according to the data dictionary, trend_pct is a percentage * 100
declining_pages = len(df[df['trend_pct'] < 0])
total_pages = len(df)
decline_rate = (declining_pages / total_pages) * 100

print(f"Total pages in starter dataset: {total_pages}")
print(f"Pages currently experiencing traffic decay (negative trend): {declining_pages}")
print(f"This means {decline_rate:.1f}% of the dataset is in active decay, justifying the need for a predictive model.")

Total pages in starter dataset: 30000
Pages currently experiencing traffic decay (negative trend): 19715
This means 65.7% of the dataset is in active decay, justifying the need for a predictive model.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

>The Decision: Which page should an editor fix first?

>The Action: A content editor will allocate their limited time to refresh or update the specific pages flagged by the model.

>The Cost of a Wrong Call: A false positive results in wasted editor hours on a page that was fine, while a false negative results in a missed decline and lost organic traffic.  

>Why ML helps: A plain rule isn't enough because the indicators of decay are too messy to write by hand, featuring many tangled and shifting signals over time.  

>*Task* Type & Target: This is a Classification task predicting a yes/no label of an observed outcome (actual traffic decline in the sealed test month of June 2026). It will be evaluated using ROC-AUC and precision/recall vs the base rate.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Checking the base rate for our classification task
# We check the baseline proxy label to understand the class imbalance

# Create the proxy decline label based on trend_direction, as discussed in the 'label trap' section.
df['is_declining_label'] = (df['trend_direction'] == 'down')

# Calculate the base rate of the proxy decline label
proxy_target_counts = df['is_declining_label'].value_counts(normalize=True) * 100
base_rate = proxy_target_counts.get(True, 0)

print(f"Base rate of the proxy decline label in starter data: {base_rate:.1f}%")
print("To prove our ML model is useful, our precision/recall must significantly beat this base rate.")
print("Note: The final target will use the observed June 2026 warehouse data instead of this proxy.")

Base rate of the proxy decline label in starter data: 54.2%
To prove our ML model is useful, our precision/recall must significantly beat this base rate.
Note: The final target will use the observed June 2026 warehouse data instead of this proxy.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Acknowledging the label trap: we cannot use trend_direction as a feature or label
# Acknowledging the percentage gotcha: ctr is x100, so 0.76 is 0.76%
mean_ctr = df['ctr'].mean()

# Acknowledging the position gotcha: 0 means 'no data', so we must exclude it to find the real average
valid_positions = df[df['avg_position'] > 0]
mean_position = valid_positions['avg_position'].mean()
missing_position_count = len(df[df['avg_position'] == 0])

print(f"The average CTR across the starter dataset is {mean_ctr:.2f}%.")
print(f"There are {missing_position_count} rows where avg_position is 0 (meaning 'no data').")
print(f"Excluding those missing rows, the true average position is {mean_position:.1f}.")

The average CTR across the starter dataset is 0.51%.
There are 1205 rows where avg_position is 0 (meaning 'no data').
Excluding those missing rows, the true average position is 17.0.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

Ans: This model will output strictly directional and decision-support results. We cannot claim to prove Google's ranking mechanics, nor can we use trend_pct as a feature since it represents a derived rule rather than a natural occurrence.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Proving the "label trap"
# The data documentation warns that is_declining_label is mathematically derived from trend_direction.

# Re-create the proxy decline label as df might have been reloaded by a previous cell
# This line is added here to ensure 'is_declining_label' exists in the DataFrame for this cell's operations.
df['is_declining_label'] = (df['trend_direction'] == 'down')

# Group by trend_direction to see how it perfectly maps to the label
label_trap_check = df.groupby('trend_direction')['is_declining_label'].value_counts()

print("Proof of the label trap:")
print(label_trap_check)
print("\nBecause 'declining' maps 100% perfectly to True and others to False, this is a derived rule.")
print("We MUST exclude trend_direction and trend_pct from our features to avoid leakage.")

Proof of the label trap:
trend_direction  is_declining_label
down             True                  16262
flat             False                  1152
new              False                  2236
stable           False                  5962
up               False                  4388
Name: count, dtype: int64

Because 'declining' maps 100% perfectly to True and others to False, this is a derived rule.
We MUST exclude trend_direction and trend_pct from our features to avoid leakage.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [14]:
#Final sanity check before moving to ML-04
expected_rows = 30000
actual_rows = len(df)

if actual_rows == expected_rows:
    print(f"Dataset verified: {actual_rows} rows loaded correctly.")
    print("Framing complete. The research question is grounded and ready for the next phase.")
else:
    print(f"Warning: Expected {expected_rows} rows, but found {actual_rows}.")

Dataset verified: 30000 rows loaded correctly.
Framing complete. The research question is grounded and ready for the next phase.


Ans: For a content editor deciding which page to fix first, we will build a classification model from the FlyRank warehouse dataset, predicting an observed future traffic decline measured by precision/recall against the base rate. A wrong call costs wasted editor hours or a missed decline. A plain rule isn't enough because the signals are tangled and shift over time. We will claim only observed, decision-support results.